# ZuraRenamebot — Google Colab Deployment

Run your Telegram rename bot from a Colab runtime. Keep the runtime connected while the bot is running.

## Before you start
1. Open **Runtime → Change runtime type** and select Python 3.
2. In the Colab left sidebar, open **Secrets** (key icon) and add these secrets. Enable notebook access for each:
   - `API_ID`
   - `API_HASH`
   - `BOT_TOKEN`
   - `DB_URL` (MongoDB connection URI)
3. Optional secret: `DB_NAME` (defaults to `zura_rename`).

Do not paste credentials into notebook cells or share a notebook containing credentials.


In [ ]:
# Clone the latest repository
import os
from pathlib import Path

REPO_URL = "https://github.com/gintokisakata918-prog/ZuraRenamebot.git"
PROJECT_DIR = Path("/content/ZuraRenamebot")

if PROJECT_DIR.exists():
    !git -C /content/ZuraRenamebot pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())


In [ ]:
# Install system tools used by optional media and RAR archive features
!apt-get update -qq
!apt-get install -y -qq ffmpeg p7zip-full

# Install Python dependencies
!pip -q install -r requirements.txt
print("Dependencies installed.")


In [ ]:
# Load credentials from Colab Secrets into environment variables
from google.colab import userdata
import os

required_secrets = ["API_ID", "API_HASH", "BOT_TOKEN", "DB_URL"]
missing = []
for key in required_secrets:
    try:
        value = userdata.get(key)
        if not value:
            missing.append(key)
        else:
            os.environ[key] = str(value)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        "Add these missing secrets in Colab's Secrets panel and enable notebook access: "
        + ", ".join(missing)
    )

try:
    os.environ["DB_NAME"] = str(userdata.get("DB_NAME") or "zura_rename")
except Exception:
    os.environ["DB_NAME"] = "zura_rename"

os.environ.setdefault("DOWNLOAD_DIR", "/content/zura_downloads")
os.environ.setdefault("PYROGRAM_WORKERS", "16")
os.environ.setdefault("MAX_CONCURRENT_TRANSMISSIONS", "4")
os.makedirs(os.environ["DOWNLOAD_DIR"], exist_ok=True)
print("Credentials loaded from Colab Secrets. Values are not displayed.")


## Start the bot

Run the next cell. It stays active while the bot is running. To stop the bot, interrupt the cell or disconnect the runtime. Colab runtimes are temporary and may disconnect due to inactivity or usage limits.

If you change code or pull new commits, stop the running bot, rerun the clone/update cell, reinstall dependencies if needed, reload secrets, and start it again.


In [ ]:
!python bot.py
